# COVID-19 Dynamics in Colombia: SIR Modeling and Temporal Regression

**Authors:** Camilo Arias and Luna Lugo  
**Academic project:** HealthTech 2, Universidad Externado de Colombia

## Objective

Analyze COVID-19 epidemiological dynamics in Colombia from a national time series, build an interpretable transmission model, and evaluate a predictive model for recent incidence.


## Problem context

The COVID-19 pandemic produced extensive time-series data on confirmed cases, deaths, recoveries, and diagnostic testing. This project uses the **COVID-19 Data Hub** level-1 dataset to describe Colombia's epidemic trajectory and to build complementary explanatory and predictive models.

The analysis combines:

- an epidemiological **SIR model** for interpretation; and
- a lag-based **linear regression model** for short-term prediction of the 7-day moving average of new cases.


# Exploratory Data Analysis (EDA)


## Libraries and configuration


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=True)


## Load the dataset

The loader checks several common local and notebook paths so the analysis can run in different environments.


In [ ]:
possible_paths = [
    Path('data/covid19datahub_level1.csv'),
    Path('../data/covid19datahub_level1.csv'),
    Path('covid19datahub_level1.csv'),
    Path('/content/covid19datahub_level1.csv'),
    Path('/mnt/data/covid19datahub_level1.csv')
]

ruta = next((str(p) for p in possible_paths if p.exists()), None)

if ruta is None:
    raise FileNotFoundError(
        "No se encontró el archivo covid19datahub_level1.csv. "
        "Sube el archivo al entorno o ajusta la ruta manualmente."
    )

df = pd.read_csv(ruta)
print("Archivo cargado desde:", ruta)
df.head()


In [ ]:
print("Dataset dimensions:", df.shape)

display(df.head())


In [ ]:
df.info()


In [ ]:

resumen_variables = pd.DataFrame({
    'variable': df.columns,
    'tipo_dato': df.dtypes.values,
    'no_nulos': df.notna().sum().values,
    'nulos': df.isna().sum().values
})

display(resumen_variables)


In [ ]:

if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    print("Converted 'date' to datetime.")
    print(df['date'].dtype)


## Data cleaning


In [ ]:
faltantes = (df.isna().mean() * 100).sort_values(ascending=False)
faltantes = faltantes.round(2)

display(faltantes.to_frame('missing_percentage').head(20))


In [ ]:
faltantes_plot = faltantes[faltantes > 0]

plt.figure(figsize=(10, 6))
faltantes_plot.head(20).sort_values().plot(kind='barh')
plt.title('Top 20 columns by missing values')
plt.xlabel('Missing values (%)')
plt.ylabel('Columna')
plt.show()


In [ ]:

df_cleann= df.copy()

print("Original dimensions:", df.shape)
print("Working-copy dimensions:", df_cleann.shape)


In [ ]:
df_cleann['date'] = pd.to_datetime(df_cleann['date'], errors='coerce')


In [ ]:
if 'iso_alpha_3' in df_cleann.columns:
    df_clean = df_cleann[df_cleann['iso_alpha_3'] == 'COL'].copy()
elif 'administrative_area_level_1' in df_cleann.columns:
    df_clean = df_cleann[df_cleann['administrative_area_level_1'].astype(str).str.upper() == 'COLOMBIA'].copy()

print("Dimensión después de filtrar Colombia:", df_clean.shape)
display(df_clean.head())


In [ ]:

columnas_utiles = [
    'date',
    'id',
    'iso_alpha_3',
    'administrative_area_level_1',
    'confirmed',
    'deaths',
    'recovered',
    'tests',
    'population'
]

columnas_utiles = [col for col in columnas_utiles if col in df_clean.columns]
df_eda = df_clean[columnas_utiles].copy()

print("Dimensión de la base reducida:", df_eda.shape)
print("Columns seleccionadas:", df_eda.columns.tolist())
display(df_eda.head())


In [ ]:
cols_numericas = ['confirmed', 'deaths', 'recovered', 'tests', 'population']
cols_numericas = [col for col in cols_numericas if col in df_eda.columns]

for col in cols_numericas:
    df_eda[col] = pd.to_numeric(df_eda[col], errors='coerce')


In [ ]:
for col in ['confirmed', 'deaths', 'recovered', 'tests']:
    if col in df_eda.columns:
        df_eda[col] = df_eda[col].ffill().fillna(0)
if 'population' in df_eda.columns:
    df_eda['population'] = df_eda['population'].ffill().bfill()
if 'iso_alpha_3' in df_eda.columns:
    df_eda['iso_alpha_3'] = df_eda['iso_alpha_3'].fillna('COL')

if 'administrative_area_level_1' in df_eda.columns:
    df_eda['administrative_area_level_1'] = df_eda['administrative_area_level_1'].fillna('Colombia')

if 'id' in df_eda.columns:
    df_eda['id'] = df_eda['id'].ffill().bfill()


In [ ]:
print("Total duplicates:", df_eda.duplicated().sum())
print("Duplicates by id and date:", df_eda.duplicated(subset=['id', 'date']).sum())


In [ ]:
for col in ['confirmed', 'deaths', 'recovered', 'tests', 'population']:
    if col in df_eda.columns:
        print(col, (df_eda[col] < 0).sum())


In [ ]:
df_eda['tests'] = df_eda['tests'].cummax()
for col in ['confirmed', 'deaths', 'recovered', 'tests']:
    if col in df_eda.columns:
        print(col, (df_eda[col].diff() < 0).sum())


In [ ]:
print("Missing dates:", df_eda['date'].isna().sum())
print("Duplicate dates:", df_eda['date'].duplicated().sum())
print("Minimum date:", df_eda['date'].min())
print("Maximum date:", df_eda['date'].max())


In [ ]:
print(df_eda['population'].nunique())
print(df_eda['population'].unique()[:10])


In [ ]:
df_eda = df_eda.sort_values('date').reset_index(drop=True)

print("Final dimensions:", df_eda.shape)
print("\nMissing values by column:")
print(df_eda.isna().sum())

display(df_eda.head())
display(df_eda.tail())
df_eda.info()


## Descriptive statistics


In [ ]:
sns.set(style="whitegrid")
pd.set_option('display.max_columns', None)

df_num = df_eda.select_dtypes(include=[np.number])

print("Numerical variables:")
print(df_num.columns.tolist())
print("\nDimensión de la matriz numérica:", df_num.shape)

display(df_num.head())


In [ ]:
estadisticas = df_num.describe().T
display(estadisticas)


The descriptive statistics show substantial dispersion across the epidemiological indicators. Because confirmed cases, deaths, recoveries, and tests are cumulative series, their values naturally increase over time and should not be interpreted as independent observations.


In [ ]:
resumen_extra = pd.DataFrame({
    'mean': df_num.mean(),
    'meann': df_num.meann(),
    'std_dev': df_num.std(),
    'variance': df_num.var(),
    'minimum': df_num.min(),
    'maximum': df_num.max(),
    'skewness': df_num.skew()
}).round(2)

display(resumen_extra)


Confirmed cases, deaths, recoveries, and tests show high variability and mild positive skewness. Population remains constant throughout the study period and therefore serves only as a reference value.


## Visual analysis


In [ ]:
sns.set(style="whitegrid")
pd.set_option('display.max_columns', None)

df_vis = df_eda.copy()
df_vis = df_vis.sort_values('date').reset_index(drop=True)

print(df_vis.shape)
display(df_vis.head())


### Boxplots of selected numerical variables


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

sns.boxplot(x=df_eda['confirmed'], ax=axes[0, 0])
axes[0, 0].set_title('Boxplot of cumulative confirmed cases')
axes[0, 0].set_xlabel('Confirmed')

sns.boxplot(x=df_eda['deaths'], ax=axes[0, 1])
axes[0, 1].set_title('Boxplot of cumulative deaths')
axes[0, 1].set_xlabel('Deaths')

sns.boxplot(x=df_eda['recovered'], ax=axes[1, 0])
axes[1, 0].set_title('Boxplot of cumulative recoveries')
axes[1, 0].set_xlabel('Recovered')

sns.boxplot(x=df_eda['tests'], ax=axes[1, 1])
axes[1, 1].set_title('Boxplot of cumulative tests')
axes[1, 1].set_xlabel('Tests')

plt.tight_layout()
plt.show()


The cumulative indicators have wide interquartile ranges but no isolated extreme observations under the standard boxplot rule. This pattern is expected because the variables encode the full progression of the pandemic rather than a stable cross-sectional distribution.


### Cumulative confirmed cases, deaths, recoveries, and tests


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_vis['date'], df_vis['confirmed'], label='Confirmed cases')
ax.plot(df_vis['date'], df_vis['deaths'], label='Deaths')
ax.plot(df_vis['date'], df_vis['recovered'], label='Recoveries')
ax.plot(df_vis['date'], df_vis['tests'], label='Tests')
ax.set_title('Colombia: cumulative confirmed cases, deaths, recoveries, and tests')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative value')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


All major cumulative indicators increase over the study period. Recoveries broadly track confirmed cases, while deaths grow at a lower magnitude. Testing capacity also expands substantially over time.


### Cumulative confirmed cases


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_vis['date'], df_vis['confirmed'])
ax.set_title('Colombia: cumulative confirmed cases')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative confirmed cases')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Confirmed cases grow continuously, with steeper segments during periods of accelerated transmission—especially around mid-2021—and a lower relative growth rate toward the end of the series.


### Cumulative deaths


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_vis['date'], df_vis['deaths'])
ax.set_title('Colombia: cumulative COVID-19 deaths')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative deaths')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Cumulative deaths rise throughout the study period. The increase is strongest from 2020 through mid-2021 and slows toward the end of the series.


### Cumulative recoveries


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_vis['date'], df_vis['recovered'])
ax.set_title('Colombia: cumulative recoveries')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative recoveries')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Recoveries rise steadily and accelerate during major transmission waves. Growth continues near the end of the series at a lower relative rate.


### Cumulative diagnostic tests


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_vis['date'], df_vis['tests'])
ax.set_title('Colombia: cumulative diagnostic tests')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative tests')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Testing expands continuously, with faster growth from the second half of 2020 and throughout 2021, reflecting increased diagnostic capacity.


### Distributions of selected numerical variables


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

sns.histplot(df_vis['confirmed'], kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Distribution of cumulative confirmed cases')
axes[0, 0].set_xlabel('Confirmed')

sns.histplot(df_vis['deaths'], kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Distribution of cumulative deaths')
axes[0, 1].set_xlabel('Deaths')

sns.histplot(df_vis['recovered'], kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Distribution of cumulative recoveries')
axes[1, 0].set_xlabel('Recovered')

sns.histplot(df_vis['tests'], kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Distribution of cumulative tests')
axes[1, 1].set_xlabel('Tests')

plt.tight_layout()
plt.show()


The cumulative indicators are not symmetrically distributed. Observations concentrate at low values early in the pandemic and at high values later, producing the skewed distributions expected from cumulative time series.


### Correlation among confirmed cases, deaths, recoveries, tests, and population


In [ ]:
corr = df_vis[['confirmed', 'deaths', 'recovered', 'tests', 'population']].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f')
plt.title('Colombia: correlation among numerical variables in the cleaned dataset')
plt.tight_layout()
plt.show()


Confirmed cases, deaths, recoveries, and tests are almost perfectly positively correlated because they increase together over time. This temporal co-movement should not be interpreted as evidence of causality. Population has no meaningful correlation because it is constant.


### Confirmed cases versus deaths


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_vis, x='confirmed', y='deaths')
plt.title('Colombia: confirmed cases versus deaths')
plt.xlabel('Cumulative confirmed cases')
plt.ylabel('Cumulative deaths')
plt.tight_layout()
plt.show()


Confirmed cases and deaths show a strong positive association. Both variables are cumulative, so their nearly linear relationship primarily reflects shared growth over time.


### Confirmed cases versus recoveries


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_vis, x='confirmed', y='recovered')
plt.title('Colombia: confirmed cases versus recoveries')
plt.xlabel('Cumulative confirmed cases')
plt.ylabel('Cumulative recoveries')
plt.tight_layout()
plt.show()


Confirmed cases and recoveries display a strong positive relationship, consistent with their cumulative construction and parallel evolution during the pandemic.


### Diagnostic tests versus confirmed cases


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_vis, x='tests', y='confirmed')
plt.title('Colombia: tests versus confirmed cases')
plt.xlabel('Cumulative tests')
plt.ylabel('Cumulative confirmed cases')
plt.tight_layout()
plt.show()


Testing and confirmed cases rise together. Greater testing capacity is associated with more detected cases, although the cumulative structure also contributes to this pattern.


### Population remains constant


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_vis['date'], df_vis['population'])
ax.set_title('Colombia: recorded population over time')
ax.set_xlabel('Date')
ax.set_ylabel('Population')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


The population field is constant throughout the series and is used as a fixed reference for the epidemiological model.


# SIR model: epidemiological parameter estimation

The cleaned Colombia series is transformed into three compartments:

- **S (Susceptible):** population minus cumulative confirmed cases;
- **I (Infectious):** confirmed cases minus recoveries and deaths; and
- **R (Removed):** recoveries plus deaths.

The transmission rate (β) and removal rate (γ) are estimated through a discrete approximation of the SIR differential equations. The model is then used to estimate the basic reproduction number, infectious period, epidemic peak, and final epidemic size.


In [ ]:

# ══════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")

N = float(df_eda['population'].iloc[0])

df_sir = df_eda[['date', 'confirmed', 'recovered', 'deaths']].copy()
df_sir = df_sir.sort_values('date').reset_index(drop=True)

df_sir['I'] = (df_sir['confirmed'] - df_sir['recovered'] - df_sir['deaths']).clip(lower=0)
df_sir['Removed'] = df_sir['recovered'] + df_sir['deaths']
df_sir['S'] = (N - df_sir['confirmed']).clip(lower=0)
df_sir['t'] = (df_sir['date'] - df_sir['date'].iloc[0]).dt.days.astype(float)

df_sir['I_s'] = df_sir['I'].rolling(7, center=True, min_periods=1).mean()
df_sir['S_s'] = df_sir['S'].rolling(7, center=True, min_periods=1).mean()
df_sir['R_s'] = df_sir['Removed'].rolling(7, center=True, min_periods=1).mean()
df_sir['conf_s'] = df_sir['confirmed'].rolling(7, center=True, min_periods=1).mean()


df_sir['new_cases']   = df_sir['conf_s'].diff().clip(lower=0)
df_sir['new_removed'] = df_sir['R_s'].diff().clip(lower=0)

mask = (df_sir['I_s'] > 5000) & (df_sir['S_s'] > 1e6)

foI = (df_sir.loc[mask, 'S_s'] * df_sir.loc[mask, 'I_s'] / N).replace(0, np.nan)

beta_serie  = (df_sir.loc[mask, 'new_cases']   / foI).replace([np.inf, -np.inf], np.nan).dropna()
gamma_serie = (df_sir.loc[mask, 'new_removed'] / df_sir.loc[mask, 'I_s'].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).dropna()

beta_fit  = float(beta_serie.meann())
gamma_fit = float(gamma_serie.meann())

beta_ci   = (float(beta_serie.quantile(0.025)),  float(beta_serie.quantile(0.975)))
gamma_ci  = (float(gamma_serie.quantile(0.025)), float(gamma_serie.quantile(0.975)))

R0 = beta_fit / gamma_fit
R0_low  = beta_ci[0] / gamma_ci[1]
R0_high = beta_ci[1] / gamma_ci[0]

infectious_period = 1.0 / gamma_fit

peak_idx  = df_sir['I'].idxmax()
peak_date = df_sir.loc[peak_idx, 'date']
peak_I    = df_sir.loc[peak_idx, 'I']
peak_day  = int(df_sir.loc[peak_idx, 't'])

total_confirmados = df_sir['confirmed'].iloc[-1]
final_size_pct    = 100.0 * total_confirmados / N

df_early = df_sir[(df_sir['t'] >= 5) & (df_sir['t'] <= 60) & (df_sir['confirmed'] > 0)].copy()
df_early['log_c'] = np.log(df_early['confirmed'].replace(0, np.nan))
df_early = df_early.dropna(subset=['log_c'])
if len(df_early) >= 5:
    r_coef      = np.polyfit(df_early['t'], df_early['log_c'], 1)
    growth_rate = r_coef[0]
    doubling_time = np.log(2) / growth_rate if growth_rate > 0 else float('nan')
else:
    growth_rate, doubling_time = float('nan'), float('nan')

separador = "═" * 68
print(separador)
print("   EPIDEMIOLOGICAL PARAMETER SUMMARY — SIR MODEL")
print("   Colombia · COVID-19 · 2020-03-05 → 2021-12-19")
print(separador)
print(f"  {'Parámetro':<38} {'Valor':>10}   {'IC 95%'}")
print("─" * 68)
print(f"  {'β  (tasa de transmisión)':<38} {beta_fit:>10.5f}   [{beta_ci[0]:.5f}, {beta_ci[1]:.5f}]")
print(f"  {'γ  (tasa de recuperación)':<38} {gamma_fit:>10.5f}   [{gamma_ci[0]:.5f}, {gamma_ci[1]:.5f}]")
print(f"  {'R₀ (número reproductivo básico)':<38} {R0:>10.3f}   [{R0_low:.3f}, {R0_high:.3f}]")
print(f"  {'Infectious period (1/γ, days)':<38} {infectious_period:>10.1f}")
print(f"  {'Infection peak — date':<38} {str(peak_date.date()):>10}")
print(f"  {'Infection peak — day no.':<38} {peak_day:>10}")
print(f"  {'Active infections at peak':<38} {peak_I:>10,.0f}")
print(f"  {'Tamaño final de epidemia (% pob.)':<38} {final_size_pct:>9.2f}%")
print(f"  {'Total confirmed cases':<38} {total_confirmados:>10,.0f}")
print(f"  {'Exponential growth rate r':<38} {growth_rate:>10.4f}  (days 5–60)")
print(f"  {'Doubling time (days)':<38} {doubling_time:>10.1f}  (early exponential phase)")
print(separador)
print()
print("Notas metodológicas:")
print("  • β y γ: meann de estimaciones diarias β_t = new_cases_t / (S_t·I_t/N)")
print("    y γ_t = nuevos_removidos_t / I_t  (suavizado 7 días, solo con I > 5 000).")
print("  • IC 95%: percentiles 2.5 y 97.5 de las estimaciones diarias.")
print("  • The 2020–2021 data cover multiple waves; β and γ are average parameters.")
print("  • Doubling time estimated by log-linear regression en days 5–60.")


In [ ]:

# ══════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════

def sir_ode(y, t, beta, gamma):
    S, I, R = y
    fuerza  = beta * S * I / N
    return [-fuerza, fuerza - gamma * I, gamma * I]

y0_sim = [
    float(df_sir['S'].iloc[0]),
    max(float(df_sir['I'].iloc[0]), 1.0),
    float(df_sir['Removed'].iloc[0])
]
t_sim = df_sir['t'].values

sol   = odeint(sir_ode, y0_sim, t_sim, args=(beta_fit, gamma_fit))
S_sim = sol[:, 0]
I_sim = sol[:, 1]
R_sim = sol[:, 2]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.fill_between(df_sir['date'], df_sir['I'] / N * 100,
                alpha=0.2, color='tomato', label='I observado (rango diario)')
ax.plot(df_sir['date'], df_sir['I_s'] / N * 100,
        color='tomato', lw=2, label='I observado (suavizado 7d)')
ax.plot(df_sir['date'], I_sim / N * 100,
        color='firebrick', lw=2, ls='--', label='I simulado (SIR)')
ax.axvline(peak_date, color='gray', ls=':', lw=1.5,
           label=f'Pico observado: {peak_date.date()}')
ax.set_title('Active infections: observed data versus SIR model')
ax.set_xlabel('Date')
ax.set_ylabel('% de la población')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=45)

ax = axes[1]
ax.plot(df_sir['date'], S_sim / N * 100,
        color='steelblue',      lw=2.5, label='S — Susceptibles')
ax.plot(df_sir['date'], I_sim / N * 100,
        color='tomato',         lw=2.5, label='I — Infectados activos')
ax.plot(df_sir['date'], R_sim / N * 100,
        color='mediumseagreen', lw=2.5, label='R — Recoveries/Removidos')
ax.set_title(
    f'Curvas SIR simuladas\n'
    f'β = {beta_fit:.4f}  |  γ = {gamma_fit:.4f}  |  R₀ = {R0:.2f}'
)
ax.set_xlabel('Date')
ax.set_ylabel('% de la población')
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=45)

plt.suptitle('Colombia — SIR model fitted to COVID-19 data (2020–2021)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nNote: the SIR model is deterministic and represents a single wave.")
print("The Colombia data cover multiple waves, por lo que el ajuste")
print("is approximate; the parameters reflect average behavior over the period.")


In [ ]:

# ══════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════

pct_range   = np.linspace(-0.20, 0.20, 200)
beta_range  = beta_fit  * (1 + pct_range)
gamma_range = gamma_fit * (1 + pct_range)

R0_vs_beta  = beta_range  / gamma_fit    # γ fijo
R0_vs_gamma = beta_fit    / gamma_range  # β fijo

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(pct_range * 100, R0_vs_beta, color='steelblue', lw=2.5)
ax.axhline(1,  color='red',  ls='--', lw=1.5, label='R₀ = 1 (umbral epidémico)')
ax.axhline(R0, color='gray', ls=':',  lw=1.2, label=f'R₀ base = {R0:.2f}')
ax.axvline(0,  color='gray', ls=':',  lw=1.0, alpha=0.6)
ax.fill_between(pct_range * 100, R0_vs_beta, 1,
                where=(R0_vs_beta >= 1), alpha=0.15, color='red',
                label='Epidemia activa  (R₀ ≥ 1)')
ax.fill_between(pct_range * 100, R0_vs_beta, 1,
                where=(R0_vs_beta < 1),  alpha=0.15, color='green',
                label='Epidemia controlada (R₀ < 1)')
ax.set_title(f'Sensibilidad de R₀ ante variación de β\n(γ fijo = {gamma_fit:.5f})')
ax.set_xlabel('Variación de β (%)')
ax.set_ylabel('R₀')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(-22, 22)

ax = axes[1]
ax.plot(pct_range * 100, R0_vs_gamma, color='darkorange', lw=2.5)
ax.axhline(1,  color='red',  ls='--', lw=1.5, label='R₀ = 1 (umbral epidémico)')
ax.axhline(R0, color='gray', ls=':',  lw=1.2, label=f'R₀ base = {R0:.2f}')
ax.axvline(0,  color='gray', ls=':',  lw=1.0, alpha=0.6)
ax.fill_between(pct_range * 100, R0_vs_gamma, 1,
                where=(R0_vs_gamma >= 1), alpha=0.15, color='red',
                label='Epidemia activa  (R₀ ≥ 1)')
ax.fill_between(pct_range * 100, R0_vs_gamma, 1,
                where=(R0_vs_gamma < 1),  alpha=0.15, color='green',
                label='Epidemia controlada (R₀ < 1)')
ax.set_title(f'Sensibilidad de R0 ante variación de γ\n(β fijo = {beta_fit:.5f})')
ax.set_xlabel('Variación de γ (%)')
ax.set_ylabel('R₀')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(-22, 22)

plt.suptitle('Sensitivity analysis: how does R₀ change when β or γ varies by ±20%?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTabla de sensibilidad — puntos seleccionados:")
print(f"  {'Variación':<10} {'R₀  (solo β varía)':<22} {'R₀  (solo γ varía)'}")
print("  " + "─" * 58)
for pct in [-0.20, -0.10, -0.05, 0.00, 0.05, 0.10, 0.20]:
    r0_b = beta_fit * (1 + pct) / gamma_fit
    r0_g = beta_fit / (gamma_fit * (1 + pct))
    flag_b = "← controlada" if r0_b < 1 else ""
    flag_g = "← controlada" if r0_g < 1 else ""
    print(f"  {pct*100:>+5.0f}%       {r0_b:<10.4f} {flag_b:<14}  {r0_g:<10.4f} {flag_g}")


### Interpretation of the SIR parameters

- **β (transmission rate)** represents the average rate of effective infectious contact.
- **γ (removal rate)** represents the daily proportion of infected individuals who recover or otherwise leave the infectious compartment.
- **R₀ = β/γ** summarizes the average transmission potential under the model assumptions.

These estimates describe an average over the analyzed period. A classical SIR model assumes homogeneous mixing and constant parameters, so it cannot fully represent multiple waves, changing variants, vaccination, or policy interventions.


# Final methodology

1. **Data selection and cleaning:** filter Colombia, parse dates, retain the main epidemiological variables, and handle missing cumulative values consistently.
2. **Exploratory analysis:** inspect distributions, missingness, temporal patterns, correlations, and data consistency.
3. **Epidemiological modeling:** estimate SIR parameters and simulate transmission dynamics.
4. **Predictive modeling:** fit a lag-based linear regression to the 7-day moving average of new cases and validate it with a chronological 80/20 split.


# Validated predictive model: linear regression for incidence

The supervised model predicts the **7-day moving average of daily new cases** (`new_cases_7d`). Lagged case and testing indicators capture the autoregressive structure of the epidemic while reducing daily reporting noise. Model performance is evaluated on the most recent 20% of observations to preserve temporal order.


In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df_model = df_eda[['date', 'confirmed', 'deaths', 'recovered', 'tests']].copy()
df_model = df_model.sort_values('date').reset_index(drop=True)

df_model['new_cases'] = df_model['confirmed'].diff().clip(lower=0).fillna(0)
df_model['new_tests'] = df_model['tests'].diff().clip(lower=0).fillna(0)

df_model['new_cases_7d'] = df_model['new_cases'].rolling(7).mean()
df_model['new_tests_7d'] = df_model['new_tests'].rolling(7).mean()

for lag in [1, 7, 14, 21]:
    df_model[f'cases7_lag{lag}'] = df_model['new_cases_7d'].shift(lag)

for lag in [1, 7]:
    df_model[f'tests7_lag{lag}'] = df_model['new_tests_7d'].shift(lag)

features = [
    'cases7_lag1',
    'cases7_lag7',
    'cases7_lag14',
    'cases7_lag21',
    'tests7_lag1',
    'tests7_lag7'
]

df_model_ready = df_model.dropna(subset=features + ['new_cases_7d']).copy()

split_idx = int(len(df_model_ready) * 0.80)
train = df_model_ready.iloc[:split_idx].copy()
test  = df_model_ready.iloc[split_idx:].copy()

X_train = train[features]
y_train = train['new_cases_7d']
X_test  = test[features]
y_test  = test['new_cases_7d']

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

test['pred_cases_7d'] = np.clip(lr_model.predict(X_test), 0, None)

mae  = mean_absolute_error(y_test, test['pred_cases_7d'])
rmse = np.sqrt(mean_squared_error(y_test, test['pred_cases_7d']))
r2   = r2_score(y_test, test['pred_cases_7d'])
mape = np.nanmean(np.abs((y_test - test['pred_cases_7d']) / np.where(y_test == 0, np.nan, y_test))) * 100

print("TRAINING RANGE:", train['date'].min().date(), "→", train['date'].max().date())
print("TEST RANGE       :", test['date'].min().date(),  "→", test['date'].max().date())
print()
print("FINAL MODEL METRICS")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

coeficientes = pd.DataFrame({
    'variable': features,
    'coeficiente': lr_model.coef_
}).sort_values('coeficiente', ascending=False)

print("\nModel coefficients:")
display(coeficientes)


In [ ]:

plt.figure(figsize=(13, 5))
plt.plot(train['date'], train['new_cases_7d'], label='Training', alpha=0.35)
plt.plot(test['date'], test['new_cases_7d'], label='Observed (test)', linewidth=2)
plt.plot(test['date'], test['pred_cases_7d'], label='Predicho (prueba)', linewidth=2, linestyle='--')
plt.title('Colombia: predicción de la incidencia (mean móvil 7d) en el conjunto de prueba')
plt.xlabel('Date')
plt.ylabel('Average new cases (7 days)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.scatter(test['new_cases_7d'], test['pred_cases_7d'], alpha=0.7)
plt.plot(
    [test['new_cases_7d'].min(), test['new_cases_7d'].max()],
    [test['new_cases_7d'].min(), test['new_cases_7d'].max()],
    linestyle='--'
)
plt.title('Observado vs. predicho — conjunto de prueba')
plt.xlabel('Observado')
plt.ylabel('Predicho')
plt.tight_layout()
plt.show()


### Predictive-model interpretation

The model performs strongly on the held-out period, indicating that recent COVID-19 incidence in Colombia is largely explained by its own temporal dynamics. The most recent case lags carry the strongest predictive signal. The results should be interpreted as in-sample-period short-horizon performance, not as evidence that the model will generalize to future variants or structural changes.


# Conclusions

1. The COVID-19 Data Hub data support a consistent national time series for Colombia covering the main epidemiological indicators.
2. The exploratory analysis identifies nonlinear cumulative growth, multiple waves, and strong temporal co-movement among the indicators.
3. The SIR model provides an interpretable average representation of transmission dynamics, while its assumptions limit its ability to reproduce multiple waves.
4. The lag-based regression achieves strong short-term fit for the 7-day moving average of new cases within the evaluation period.
5. Combining mechanistic and predictive approaches provides both epidemiological interpretation and practical forecasting capability.


# Study limitations

- Official case and testing data may contain reporting delays, undercounting, or changes in diagnostic criteria.
- Strong relationships among cumulative variables may reflect common time trends rather than causality.
- The classical SIR model assumes homogeneous mixing, constant parameters, and a single epidemic wave.
- The regression excludes mobility, vaccination, variants, and policy measures.
- Predictive performance may deteriorate outside the observed period or during abrupt structural changes.


# Export files for BI tools

The next cell creates two analysis-ready CSV files:

- `healtech_powerbi_series.csv`: observed and predicted time-series indicators;
- `healtech_model_metrics.csv`: summary metrics for the final model.


In [ ]:

powerbi_series = df_model_ready[['date', 'confirmed', 'deaths', 'recovered', 'tests', 'new_cases', 'new_cases_7d']].copy()
powerbi_series = powerbi_series.merge(
    test[['date', 'pred_cases_7d']],
    on='date',
    how='left'
)

powerbi_series.to_csv('healtech_powerbi_series.csv', index=False)

metricas_modelo = pd.DataFrame({
    'metrica': ['MAE', 'RMSE', 'R2', 'MAPE'],
    'valor': [mae, rmse, r2, mape]
})

metricas_modelo.to_csv('healtech_metricas_modelo.csv', index=False)

print("Archivos exportados correctamente:")
print("- healtech_powerbi_series.csv")
print("- healtech_metricas_modelo.csv")


# Interactive HTML dashboard

The final section creates a self-contained HTML dashboard using Plotly and Seaborn. It summarizes key epidemiological indicators, temporal patterns, and predictive-model performance.


In [ ]:

import io
import base64
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# =========================
dashboard_df = powerbi_series.copy()
dashboard_df["date"] = pd.to_datetime(dashboard_df["date"])
dashboard_df = dashboard_df.sort_values("date").reset_index(drop=True)

dashboard_df["new_tests"] = dashboard_df["tests"].diff().clip(lower=0)
dashboard_df["new_deaths"] = dashboard_df["deaths"].diff().clip(lower=0)
dashboard_df["year"] = dashboard_df["date"].dt.year.astype(str)
dashboard_df["month_num"] = dashboard_df["date"].dt.month

MESES_ES = {
    1: "ene", 2: "feb", 3: "mar", 4: "abr", 5: "may", 6: "jun",
    7: "jul", 8: "ago", 9: "sep", 10: "oct", 11: "nov", 12: "dic"
}
dashboard_df["month_label"] = dashboard_df["month_num"].map(MESES_ES)

month_order = ["ene", "feb", "mar", "abr", "may", "jun",
               "jul", "ago", "sep", "oct", "nov", "dic"]

test_df = dashboard_df[dashboard_df["pred_cases_7d"].notna()].copy()

last_row = dashboard_df.iloc[-1]
peak_row = dashboard_df.loc[dashboard_df["new_cases_7d"].idxmax()]
r2_val = float(metricas_modelo.loc[metricas_modelo["metrica"] == "R2", "valor"].iloc[0])
mape_val = float(metricas_modelo.loc[metricas_modelo["metrica"] == "MAPE", "valor"].iloc[0])

def fecha_bonita(ts):
    ts = pd.to_datetime(ts)
    return f"{ts.day:02d} {MESES_ES[ts.month]} {ts.year}"

fecha_inicio = pd.to_datetime(dashboard_df["date"].min())
fecha_fin = pd.to_datetime(dashboard_df["date"].max())

# =========================
# =========================
fig_acum = make_subplots(specs=[[{"secondary_y": True}]])

for name, col in [
    ("Cumulative confirmed cases", "confirmed"),
    ("Cumulative recoveries", "recovered"),
    ("Cumulative deaths", "deaths")
]:
    fig_acum.add_trace(
        go.Scatter(
            x=dashboard_df["date"],
            y=dashboard_df[col],
            mode="lines",
            name=name
        ),
        secondary_y=False
    )

fig_acum.add_trace(
    go.Scatter(
        x=dashboard_df["date"],
        y=dashboard_df["tests"],
        mode="lines",
        name="Cumulative tests"
    ),
    secondary_y=True
)

fig_acum.update_layout(
    title=dict(
        text="Cumulative evolution of epidemiological indicators",
        x=0.03,
        xanchor="left"
    ),
    template="plotly_white",
    height=430,
    margin=dict(l=70, r=70, t=80, b=60),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    )
)

fig_acum.update_yaxes(
    title_text="Casos / recuperados / muertes",
    secondary_y=False,
    separatethousands=True
)
fig_acum.update_yaxes(
    title_text="Cumulative tests",
    secondary_y=True,
    separatethousands=True
)

# =========================
# =========================
fig_inc = go.Figure()

fig_inc.add_trace(
    go.Bar(
        x=dashboard_df["date"],
        y=dashboard_df["new_cases"],
        name="Daily new cases",
        hovertemplate="<b>%{x|%d %b %Y}</b><br>Daily cases: %{y:,.0f}<extra></extra>"
    )
)

fig_inc.add_trace(
    go.Scatter(
        x=dashboard_df["date"],
        y=dashboard_df["new_cases_7d"],
        mode="lines",
        name="Media móvil 7 días",
        hovertemplate="<b>%{x|%d %b %Y}</b><br>Media móvil 7d: %{y:,.0f}<extra></extra>"
    )
)

fig_inc.update_layout(
    title=dict(
        text="Incidencia diaria y suavizada",
        x=0.03,
        xanchor="left"
    ),
    template="plotly_white",
    height=430,
    margin=dict(l=70, r=40, t=80, b=60),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    )
)

fig_inc.update_yaxes(
    title_text="Casos",
    separatethousands=True
)

# =========================
# =========================
fig_pred = go.Figure()

fig_pred.add_trace(
    go.Scatter(
        x=dashboard_df["date"],
        y=dashboard_df["new_cases_7d"],
        mode="lines",
        name="Observado",
        hovertemplate="<b>%{x|%d %b %Y}</b><br>Observado: %{y:,.0f}<extra></extra>"
    )
)

fig_pred.add_trace(
    go.Scatter(
        x=test_df["date"],
        y=test_df["pred_cases_7d"],
        mode="lines",
        line=dict(dash="dash"),
        name="Predicho (prueba)",
        hovertemplate="<b>%{x|%d %b %Y}</b><br>Predicho: %{y:,.0f}<extra></extra>"
    )
)

if not test_df.empty:
    fig_pred.add_vrect(
        x0=test_df["date"].min(),
        x1=test_df["date"].max(),
        fillcolor="lightgray",
        opacity=0.22,
        line_width=0,
        annotation_text="Ventana de prueba",
        annotation_position="top left"
    )

fig_pred.update_layout(
    title=dict(
        text="Final model: observed versus predicted",
        x=0.03,
        xanchor="left"
    ),
    template="plotly_white",
    height=430,
    margin=dict(l=70, r=40, t=80, b=60),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    )
)

fig_pred.update_yaxes(
    title_text="Casos (mean móvil 7d)",
    separatethousands=True
)

# =========================
# =========================
monthly = (
    dashboard_df.assign(month_start=dashboard_df["date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month_start", as_index=False)
    .agg(
        avg_new_cases_7d=("new_cases_7d", "mean"),
        total_new_cases=("new_cases", "sum"),
        total_new_deaths=("new_deaths", "sum")
    )
)

fig_month = make_subplots(specs=[[{"secondary_y": True}]])

fig_month.add_trace(
    go.Bar(
        x=monthly["month_start"],
        y=monthly["total_new_cases"],
        name="Monthly new cases",
        hovertemplate="<b>%{x|%b %Y}</b><br>Cases: %{y:,.0f}<extra></extra>"
    ),
    secondary_y=False
)

fig_month.add_trace(
    go.Scatter(
        x=monthly["month_start"],
        y=monthly["avg_new_cases_7d"],
        mode="lines+markers",
        name="Promedio mean móvil 7d",
        hovertemplate="<b>%{x|%b %Y}</b><br>Promedio 7d: %{y:,.0f}<extra></extra>"
    ),
    secondary_y=True
)

fig_month.update_layout(
    title=dict(
        text="Resumen mensual de la pandemia",
        x=0.03,
        xanchor="left"
    ),
    template="plotly_white",
    height=460,
    margin=dict(l=70, r=70, t=80, b=70),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    ),
    bargap=0.25
)

fig_month.update_xaxes(
    title_text="Mes",
    tickformat="%b<br>%Y",
    tickangle=0,
    showgrid=False
)

fig_month.update_yaxes(
    title_text="New cases",
    secondary_y=False,
    rangemode="tozero",
    separatethousands=True
)

fig_month.update_yaxes(
    title_text="Promedio 7d",
    secondary_y=True,
    rangemode="tozero",
    separatethousands=True
)

# =========================
# =========================
corr_cols = ["confirmed", "deaths", "recovered", "tests", "new_cases", "new_cases_7d"]
corr = dashboard_df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="Blues", fmt=".2f")
plt.title("Correlation among main indicators")
plt.tight_layout()

buf1 = io.BytesIO()
plt.savefig(buf1, format="png", dpi=180, bbox_inches="tight")
plt.close()
heatmap_b64 = base64.b64encode(buf1.getvalue()).decode("utf-8")

# =========================
# =========================
plt.figure(figsize=(12, 5))
sns.boxplot(
    data=dashboard_df,
    x="month_label",
    y="new_cases",
    hue="year",
    order=month_order
)
plt.title("Monthly distribution of daily new cases")
plt.xlabel("Mes")
plt.ylabel("New cases")
plt.xticks(rotation=0)
plt.tight_layout()

buf2 = io.BytesIO()
plt.savefig(buf2, format="png", dpi=180, bbox_inches="tight")
plt.close()
boxplot_b64 = base64.b64encode(buf2.getvalue()).decode("utf-8")

# =========================
# =========================
cards_html = f"""
<div class="cards">
  <div class="card">
    <h3>Cumulative cases</h3>
    <p>{int(last_row['confirmed']):,}</p>
  </div>

  <div class="card">
    <h3>Cumulative deaths</h3>
    <p>{int(last_row['deaths']):,}</p>
  </div>

  <div class="card">
    <h3>Pico mean móvil 7d</h3>
    <p>{peak_row['new_cases_7d']:.0f}</p>
    <span>{fecha_bonita(peak_row['date'])}</span>
  </div>

  <div class="card">
    <h3>Final model R²</h3>
    <p>{r2_val:.3f}</p>
  </div>

  <div class="card">
    <h3>MAPE</h3>
    <p>{mape_val:.2f}%</p>
  </div>

  <div class="card card-periodo">
    <h3>Periodo analizado</h3>
    <div class="periodo-wrap">
      <div class="periodo-item">
        <span class="periodo-label">Desde</span>
        <div class="periodo-fecha">{fecha_bonita(fecha_inicio)}</div>
      </div>
      <div class="periodo-separador">→</div>
      <div class="periodo-item">
        <span class="periodo-label">Hasta</span>
        <div class="periodo-fecha">{fecha_bonita(fecha_fin)}</div>
      </div>
    </div>
  </div>
</div>
"""

# =========================
# =========================
html = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<title>Dashboard Healtech COVID-19 Colombia</title>
<style>
body {{
    font-family: Arial, Helvetica, sans-serif;
    margin: 0;
    background: #f5f7fb;
    color: #1f2937;
}}

.container {{
    width: 92%;
    margin: 24px auto 40px auto;
}}

h1 {{
    margin-bottom: 6px;
    color: #163a70;
}}

h2 {{
    margin-top: 32px;
    color: #163a70;
}}

h3 {{
    margin-top: 0;
}}

.subtitle {{
    margin-top: 0;
    color: #4b5563;
}}

.cards {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
    gap: 16px;
    margin: 22px 0 28px 0;
}}

.card {{
    background: white;
    border-radius: 16px;
    padding: 18px 20px;
    box-shadow: 0 4px 14px rgba(0,0,0,0.08);
    min-height: 110px;
}}

.card h3 {{
    margin: 0 0 10px 0;
    font-size: 15px;
    color: #374151;
}}

.card p {{
    margin: 0;
    font-size: 28px;
    font-weight: 700;
    color: #163a70;
    line-height: 1.1;
    word-break: normal;
    overflow-wrap: break-word;
}}

.card span {{
    display: block;
    margin-top: 8px;
    color: #6b7280;
    font-size: 12px;
}}

.card-periodo {{
    background: linear-gradient(135deg, #ffffff 0%, #f8fbff 100%);
    border: 1px solid #dbe7f5;
}}

.periodo-wrap {{
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 10px;
    margin-top: 6px;
    flex-wrap: wrap;
}}

.periodo-item {{
    flex: 1 1 120px;
}}

.periodo-label {{
    display: block;
    font-size: 12px;
    color: #6b7280;
    margin-bottom: 4px;
}}

.periodo-fecha {{
    font-size: 22px;
    font-weight: 700;
    color: #163a70;
    line-height: 1.2;
}}

.periodo-separador {{
    font-size: 24px;
    font-weight: 700;
    color: #9ca3af;
    padding: 0 4px;
}}

.grid {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 20px;
    align-items: start;
}}

.panel {{
    background: white;
    border-radius: 16px;
    padding: 16px;
    box-shadow: 0 4px 14px rgba(0,0,0,0.08);
    margin-bottom: 20px;
}}

.img-panel img {{
    width: 100%;
    border-radius: 10px;
}}

@mean (max-width: 1000px) {{
    .grid {{
        grid-template-columns: 1fr;
    }}
}}

@mean (max-width: 700px) {{
    .periodo-wrap {{
        flex-direction: column;
        align-items: flex-start;
    }}

    .periodo-separador {{
        display: none;
    }}

    .periodo-fecha {{
        font-size: 20px;
    }}
}}
</style>
</head>
<body>
<div class="container">
  <h1>Dashboard Healtech · COVID-19 en Colombia</h1>
  <p class="subtitle">Visual summary of the final project: exploratory analysis, epidemiological evolution, and predictive-model validation.</p>

  {cards_html}

  <div class="grid">
    <div class="panel">{fig_acum.to_html(full_html=False, include_plotlyjs='cdn')}</div>
    <div class="panel">{fig_inc.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="panel">{fig_pred.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="panel">{fig_month.to_html(full_html=False, include_plotlyjs=False)}</div>
  </div>

  <h2>Visualizaciones analíticas complementarias</h2>

  <div class="grid">
    <div class="panel img-panel">
      <h3>Correlation heatmap</h3>
      <img src="data:image/png;base64,{heatmap_b64}" alt="Heatmap">
    </div>

    <div class="panel img-panel">
      <h3>Monthly distribution of new cases</h3>
      <img src="data:image/png;base64,{boxplot_b64}" alt="Boxplot">
    </div>
  </div>
</div>
</body>
</html>
"""

with open("healtech_dashboard_interactivo.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Dashboard HTML creado: healtech_dashboard_interactivo.html")
